In [ ]:
# paths
import numpy as np
import pandas as pd
from scipy import stats
import sys

sys.path.append('helpers/pcca_fa')
import helpers.pcca_fa.pcca_fa_mdl as pf
from dual_pfc_funcs import getParams, load_dict

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.style.use('scifigs.mplstyle')
SAVE_FIG = False

# parameters
params = getParams()
subjects = params['subjects']
plot_sym = params['markers']
color_map = params['color_map']

In [ ]:
data_path = 'preprocessed_data/'

pccafa = {}
for sub in subjects:
    pccafa = {**pccafa, **load_dict(data_path + sub + '_pccafa_cv15dim.pkl')}
fnames = pccafa.keys()

df = pd.DataFrame(columns=['SessionName','SvW1','SvW2','SvL1','SvL2','DSharedW1','DSharedW2','DSharedL1','DSharedL2'])
for i, (sess, curr_dat) in enumerate(pccafa.items()):
    mdl = pf.pcca_fa()
    mdl.set_params(curr_dat['params'])
    d_shared = mdl.compute_dshared()
    psv = mdl.compute_psv()
    
    df2 = {'SessionName':sess,
           'SvW1':psv['avg_psv_W_1'],'SvW2':psv['avg_psv_W_2'],'SvL1':psv['avg_psv_L_1'],'SvL2':psv['avg_psv_L_2'],
           'DSharedW1':d_shared['dshared_W_1'],'DSharedW2':d_shared['dshared_W_2'],'DSharedL1':d_shared['dshared_L_1'],'DSharedL2':d_shared['dshared_L_2']}
    df.loc[len(df)] = df2
# df

In [ ]:
# example session
ex_sess = 'Pe180726'
ex_idx = [s == ex_sess for s in df['SessionName']]
print(df[ex_idx][['SessionName','SvW1','SvW2','SvL1','SvL2']])

p = pccafa[ex_sess]['params']
mdl = pf.pcca_fa()
mdl.set_params(p)
W_1,W_2,L_1,L_2 = mdl.get_loading_matrices()

across_x1 = np.diag(W_1.dot(W_1.T))
across_x2 = np.diag(W_2.dot(W_2.T))
within_x1 = np.diag(L_1.dot(L_1.T))
within_x2 = np.diag(L_2.dot(L_2.T))

total_x1 = across_x1 + within_x1 + p['psi_1'] # WWT + LLT + Psi
total_x2 = across_x2 + within_x2 + p['psi_2'] # WWT + LLT + Psi

n1 = len(total_x1) # number of neurons in area 1 - LEFT
n2 = len(total_x2) # number of neurons in area 2 - RIGHT

d = p['d']
d1 = p['d1']
d2 = p['d2']

# plot
fig,ax = plt.subplots(2,1,sharex=True)
fig.set_figheight(2*fig.get_figheight())

n_plot = 14
idx1 = np.arange(n_plot)
ax[1].barh(np.arange(1,n_plot+1,1),total_x1[idx1],color=color_map['independent'],label='independent')
ax[1].barh(np.arange(1,n_plot+1,1),within_x1[idx1]+across_x1[idx1],color=color_map['within2'],label='within')
ax[1].barh(np.arange(1,n_plot+1,1),across_x1[idx1],color=color_map['across'],label='across')

idx2 = np.arange(18,18+n_plot,1)
ax[0].barh(np.arange(1,n_plot+1,1),total_x2[idx2],color=color_map['independent'],label='independent')
ax[0].barh(np.arange(1,n_plot+1,1),within_x2[idx2]+across_x2[idx2],color=color_map['within1'],label='within')
ax[0].barh(np.arange(1,n_plot+1,1),across_x2[idx2],color=color_map['across'],label='across')

# plot formatting
ticks = np.arange(0,n_plot,10)
ticks[0] = 1
ax[0].invert_yaxis(),ax[1].invert_yaxis()
ax[0].set_yticks(ticks),ax[1].set_yticks(ticks)
ax[0].legend(loc='lower right',frameon=False),ax[1].legend(loc='lower right',frameon=False)
ax[1].set_xticks([])
ax[1].set_ylabel('left PFC neurons',color=color_map['within2'])
ax[0].set_ylabel('right PFC neurons',color=color_map['within1'])
ax[1].spines['bottom'].set_visible(False),ax[1].spines['bottom'].set_visible(False)

if SAVE_FIG:
    pdf = PdfPages('figs/partitions_ex_sess.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# create graphics 
fig,ax = plt.subplots(1,2)
fig.set_figwidth(2*fig.get_figwidth())

jitters = [-0.1,0,0.1]
for sub,sym,jit in zip(subjects,plot_sym,jitters):
    sub_prefix = sub[:2].title()
    filt = [sub_prefix in s for s in df['SessionName']]
    tmp_df = df[filt]

    psv_within = np.concatenate((tmp_df['SvL1'], tmp_df['SvL2']))
    psv_across = np.concatenate((tmp_df['SvW1'], tmp_df['SvW2']))
    ax[0].plot(psv_within,psv_across,marker=sym,ls='',color='k',label='Monkey {:s}'.format(sub_prefix),markersize=5,fillstyle='none')

    ds_within = np.concatenate((tmp_df['DSharedL1'], tmp_df['DSharedL2']))
    ds_across = np.concatenate((tmp_df['DSharedW1'], tmp_df['DSharedW2']))
    count_dict = dict()
    for tmp in zip(ds_within,ds_across):
        if tmp not in count_dict:
            count_dict[tmp] = 1
        else:
            count_dict[tmp] += 1
    for t in count_dict:
        ax[1].plot(t[0]+jit,t[1]+jit,marker=sym,ls='',color='k',markersize=2+1.5*count_dict[t],fillstyle='none')

# add example session coloring - pepe
ax[0].plot(df[ex_idx]['SvL1'],df[ex_idx]['SvW1'],marker='o',ls='',color=color_map['within2'],markersize=5,fillstyle='none')
ax[0].plot(df[ex_idx]['SvL2'],df[ex_idx]['SvW2'],marker='o',ls='',color=color_map['within1'],markersize=5,fillstyle='none')

# make plots pretty and display them
ax[0].plot([0,40],[0,40],'k--')
ax[1].plot([0,25],[0,25],'k--')

ax[0].set_xlabel('within-area %sv', color=color_map['within'])
ax[0].set_ylabel('across-area %sv', color=color_map['across'])
ax[1].set_xlabel('within-area $d_{shared}$', color=color_map['within'])
ax[1].set_ylabel('across-area $d_{shared}$', color=color_map['across'])

ax[0].set_xlim([0,31])
ax[0].set_ylim([0,31])
ax[0].set_xticks(np.arange(0,31,15))
ax[0].set_yticks(np.arange(0,31,15))

ax[1].set_xlim([0,16])
ax[1].set_ylim([0,16])
ax[1].set_xticks(np.arange(0,16,5))
ax[1].set_yticks(np.arange(0,16,5))

ax[0].set_aspect('equal')
ax[1].set_aspect('equal')
ax[0].legend(loc='lower right')

if SAVE_FIG:
    pdf = PdfPages('figs/across_vs_within.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# statistics
alpha = 0.01

# test if across and within are different across all monkeys
psv_within = np.concatenate((df['SvL1'], df['SvL2']))
psv_across = np.concatenate((df['SvW1'], df['SvW2']))

ds_within = np.concatenate((df['DSharedL1'], df['DSharedL2']))
ds_across = np.concatenate((df['DSharedW1'], df['DSharedW2']))

print('dim: across={:.3f} +/- {:.3f} s.e.m., within={:.3f} +/- {:.3f} s.e.m.'.format(np.mean(ds_across),stats.sem(ds_across),np.mean(ds_within),stats.sem(ds_within)))
print('psv: across={:.3f} +/- {:.3f} s.e.m., within={:.3f} +/- {:.3f} s.e.m.'.format(np.mean(psv_across),stats.sem(psv_across),psv_within.mean(),stats.sem(psv_within)))

_,pdim = stats.ttest_rel(ds_across,ds_within,alternative='greater')
_,ppsv = stats.ttest_rel(psv_across,psv_within,alternative='greater')
print()
print('Across greater than within?')
print('Pooled, dim: {}, p = {:.1E}'.format(pdim<alpha,pdim))
print('Pooled, psv: {}, p = {:.1E}'.format(ppsv<alpha,ppsv))
print()
for sub in subjects:
    sub_prefix = sub[:2].title()
    filt = [sub_prefix in s for s in df['SessionName']]
    tmp_df = df[filt]

    psv_within = np.concatenate((tmp_df['SvL1'], tmp_df['SvL2']))
    psv_across = np.concatenate((tmp_df['SvW1'], tmp_df['SvW2']))

    ds_within = np.concatenate((tmp_df['DSharedL1'], tmp_df['DSharedL2']))
    ds_across = np.concatenate((tmp_df['DSharedW1'], tmp_df['DSharedW2']))

    # test if across and within are different for this monkey
    _,pdim = stats.ttest_rel(ds_across,ds_within,alternative='greater')
    _,ppsv = stats.ttest_rel(psv_across,psv_within,alternative='greater')
    print('Monkey {:s}, dim: {}, p = {:.1E}'.format(sub_prefix,pdim<alpha,pdim))
    print('Monkey {:s}, psv: {}, p = {:.1E}'.format(sub_prefix,ppsv<alpha,ppsv))

In [ ]:
# difference histogram plot - insets
psv_within = np.concatenate((df['SvL1'], df['SvL2']))
psv_across = np.concatenate((df['SvW1'], df['SvW2']))

ds_within = np.concatenate((df['DSharedL1'], df['DSharedL2']))
ds_across = np.concatenate((df['DSharedW1'], df['DSharedW2']))

d_diff = ds_within - ds_across
psv_diff = psv_within - psv_across

fig,ax = plt.subplots(1,2)
fig.set_figwidth(2*fig.get_figwidth())

tmp = ax[1].plot(d_diff.mean(),22,marker='v',color=[.5,.5,.5],ms=12)
ax[1].hist(d_diff,bins=10,color=tmp[0].get_color())
ax[1].plot([0,0],ax[1].get_ylim(),'k--')
ax[1].set_title('d_shared')

tmp = ax[0].plot(psv_diff.mean(),12,marker='v',color=[.5,.5,.5],ms=12)
ax[0].hist(psv_diff,bins=18,color=tmp[0].get_color())
ax[0].plot([0,0],ax[0].get_ylim(),'k--')
ax[0].set_title('%sv')

print('%sv diff - ', psv_diff.mean())
print('d-shared diff - ', d_diff.mean())

if SAVE_FIG:
    pdf = PdfPages('figs/across_vs_within_diffs.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()